In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:

PATH = '/Users/maraeckart/dev/hslu/fs26/DSPRO/data_testing/glamos_data/massbalance_2025_r2025/'

path_01 = 'massbalance_fixdate_2025_r2025.csv'
path_02 = 'massbalance_fixdate_elevationbins_2025_r2025.csv'
path_03 = 'massbalance_observation_2025_r2025.csv'
path_04 = 'massbalance_observation_elevationbins_2025_r2025.csv'



def load_glamos(file):
    df = pd.read_csv(
        file,
        skiprows=6,
        engine="python",
        on_bad_lines="skip"
    )
    return df

df_01 = load_glamos(PATH + path_01)
df_02 = load_glamos(PATH + path_02)
df_03 = load_glamos(PATH + path_03)
df_04 = load_glamos(PATH + path_04)

df_02.head()


In [ ]:
df_02.head()

In [ ]:
df_03.head()

In [ ]:
df_04.head()

In [ ]:
print(df_01.shape)
print(df_02.shape)
print(df_03.shape)
print(df_04.shape)

In [ ]:
df_01 = df_01.iloc[2:].reset_index(drop=True)
df_02 = df_02.iloc[2:].reset_index(drop=True)
df_03 = df_03.iloc[2:].reset_index(drop=True)
df_04 = df_04.iloc[2:].reset_index(drop=True)

In [ ]:
print(df_01.columns.tolist())
print(df_02.columns.tolist())

In [ ]:
df_03['end date of observation'] = pd.to_datetime(df_03['end date of observation'])
df_03['year'] = df_03['end date of observation'].dt.year
df_03['annual mass balance'] = pd.to_numeric(df_03['annual mass balance'])

In [ ]:
print(df_01.dtypes)
print(df_02.dtypes)
print(df_03.dtypes)
print(df_04.dtypes)

In [ ]:
date_cols = [
    'start date of observation',
    'end date of winter observation',
    'end date of observation'
]

for df in [df_01, df_02, df_03, df_04]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
for df in [df_01, df_02, df_03, df_04]:
    df[df.columns.difference([
        'glacier name',
        'glacier id',
        'observer',
        'start date of observation',
        'end date of winter observation',
        'end date of observation'
    ])] = df[df.columns.difference([
        'glacier name',
        'glacier id',
        'observer',
        'start date of observation',
        'end date of winter observation',
        'end date of observation'
    ])].apply(pd.to_numeric, errors='coerce')

In [ ]:
df_03.info()
df_03.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.bar(df_03['year'], df_03['annual mass balance'])
plt.axhline(0)
plt.xlabel('Year')
plt.ylabel('Annual mass balance')
plt.show()

In [ ]:
def plot_observations_per_glacier(df, title):
    counts = df.groupby('glacier name').size()
    
    counts.plot(kind='bar', figsize=(10,5))
    plt.ylabel('Number of observations')
    plt.title(title)
    plt.xticks(rotation=90)
    plt.show()

In [ ]:
plot_observations_per_glacier(df_01, 'Observations per glacier (fixdate)')
plot_observations_per_glacier(df_02, 'Observations per glacier (fixdate elevation bins)')
plot_observations_per_glacier(df_03, 'Observations per glacier (observation)')
plot_observations_per_glacier(df_04, 'Observations per glacier (observation elevation bins)')

In [ ]:
for df in [df_01, df_02, df_03, df_04]:
    df['end date of observation'] = pd.to_datetime(df['end date of observation'], errors='coerce')
    df['year'] = df['end date of observation'].dt.year

In [ ]:
def plot_observations_per_year(df, title):

    counts = df.groupby('year').size()

    fig, ax = plt.subplots(figsize=(10,5))

    counts.plot(kind='bar', ax=ax, color="#4C72B0")

    ax.set_ylabel("Number of observations")
    ax.set_title(title)

    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_observations_per_year(df_01, 'Observations per year (fixdate)')
plot_observations_per_year(df_02, 'Observations per year (fixdate elevation bins)')
plot_observations_per_year(df_03, 'Observations per year (observation)')
plot_observations_per_year(df_04, 'Observations per year (observation elevation bins)')

In [ ]:
def plot_observations_per_decade(df, title):
    df_plot = df.copy()
    df_plot['decade'] = (df_plot['year'] // 10) * 10

    counts = df_plot.groupby('decade').size()

    fig, ax = plt.subplots(figsize=(10, 5))
    counts.plot(kind='bar', ax=ax, width=0.8)

    ax.set_title(title)
    ax.set_xlabel('Decade')
    ax.set_ylabel('Number of observations')

    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_observations_per_decade(df_04,'Observations per Decade')

In [ ]:
counts = df_04.groupby(['glacier name', 'year']).size().reset_index(name='observations')
counts.head()

In [ ]:
glaciers = counts['glacier name'].unique()

fig, axes = plt.subplots(len(glaciers), 1, figsize=(10, 3*len(glaciers)), sharex=True)

for ax, glacier in zip(axes, glaciers):
    
    data = counts[counts['glacier name'] == glacier]
    
    ax.bar(data['year'], data['observations'])
    
    ax.set_title(glacier)
    ax.set_ylabel("Obs")
    
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[-1].set_xlabel("Year")

plt.tight_layout()
plt.show()

In [ ]:
import math

n = len(glaciers)
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), sharex=True)

axes = axes.flatten()

for ax, glacier in zip(axes, glaciers):
    
    data = counts[counts['glacier name'] == glacier]
    
    ax.bar(data['year'], data['observations'])
    ax.set_title(glacier)
    
    ax.spines[['top','right']].set_visible(False)

for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
df_01['dataset'] = 'fixdate'
df_02['dataset'] = 'fixdate_elevationbins'
df_03['dataset'] = 'observation'
df_04['dataset'] = 'observation_elevationbins'

In [ ]:
combined = pd.concat([df_01, df_02, df_03, df_04], ignore_index=True)

In [ ]:
count_table = (
    combined
    .groupby(['dataset', 'glacier name', 'year'])
    .size()
    .reset_index(name='observations')
)

In [ ]:
count_per_glacier = (
    combined
    .groupby(['glacier name', 'year'])
    .size()
    .reset_index(name='observations')
)

count_per_glacier.head()

In [ ]:
glaciers = count_per_glacier['glacier name'].unique()
n = len(glaciers)
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows), sharex=True)

axes = axes.flatten()

for ax, glacier in zip(axes, glaciers):
    
    data = count_per_glacier[count_per_glacier['glacier name'] == glacier]
    
    ax.bar(data['year'], data['observations'])
    ax.set_title(glacier)
    
    ax.spines[['top','right']].set_visible(False)

for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()